# SoundStream Demo

In [8]:
audio_url = 'https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav' # insert your link here

In [ ]:
# %pip install -q condacolab
# import condacolab
# condacolab.install()
# !conda create -n ss python=3.11 -y

### Cloning repo

In [ ]:
import os

if not os.path.exists('src'):
    !git clone https://github.com/arslan-yam/SoundStream.git
    %cd SoundStream

### Installing dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q torchcodec gdown
!pip install gdown
!pip install IPython

import sys
import subprocess

if sys.platform.startswith('linux'):
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], check=True)
    %pip install -q torchcodec
    
import gdown
import torch, torchaudio
from IPython.display import Audio, display
import urllib.request

### Downloading weights

In [ ]:
os.makedirs('checkpoints', exist_ok=True)
best_model_path = 'checkpoints/model_best.pth'
file_id = "1UchYZOlm3cDxykMwOOhPObgGnGKJdPu-"
gdown.download(id=file_id, output=best_model_path, quiet=False)

### Re-synthesizing your audio

In [ ]:
from src.model import SoundStreamModel

urllib.request.urlretrieve(audio_url, 'sample.wav')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SoundStreamModel().to(device).eval()
checkpoint = torch.load('checkpoints/model_best.pth', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint, strict=False)

wav, sr = torchaudio.load('sample.wav')
if sr != 16000:
    wav = torchaudio.functional.resample(wav, sr, 16000)
    
wav = wav.mean(dim=0, keepdim=True).unsqueeze(0).to(device)
with torch.no_grad():
    out = model(audio=wav)
pred = out['audio_pred'].squeeze(0).cpu().clamp(-1, 1)

print('original:')
display(Audio(wav.squeeze(0).cpu().numpy(), rate=16000))
print('re-synthesized:')
display(Audio(pred.numpy(), rate=16000))